In [4]:
import os
from pathlib import Path
import pandas as pd
import requests

# Look for .env in the parent directory if your notebook is in a subfolder (e.g., notebooks/)
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
for line in env_path.read_text().splitlines() if env_path.exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())

API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
API_KEY = os.environ.get("RAWTREE_API_KEY")
TEAM = os.environ.get("TEAM_ID")

if not API_KEY:
    raise ValueError("RAWTREE_API_KEY is missing! Check your .env file location.")

http = requests.Session()
http.headers["Authorization"] = f"Bearer {API_KEY}"

In [5]:
def query(sql: str) -> pd.DataFrame:
    """Helper function to execute SQL queries against the RawTree API and return a DataFrame."""
    r = http.post(f"{API}/v1/query", json={"sql": sql}, timeout=600)
    r.raise_for_status()
    data = r.json().get("data", [])
    return pd.DataFrame(data)

print("Fetching all prediction data sources from RawTree...")

Fetching all prediction data sources from RawTree...


In [6]:
# All-time best sellers ranking (derived from historical purchases)
sales_sql = """
SELECT product_id, count() AS n
FROM shop.history_events
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY n DESC
"""
best_df = query(sales_sql)
BEST_SELLERS: list[str] = best_df["product_id"].tolist() if not best_df.empty else []

# Current out-of-stock items (recommending these makes an answer invalid)
stock_sql = """
SELECT product_id::String AS product_id
FROM shop.catalog_updates
WHERE change::String IN ('out_of_stock', 'back_in_stock')
GROUP BY product_id
HAVING argMax(change::String, ts::DateTime64(3)) = 'out_of_stock'
"""
stock_df = query(stock_sql)
OUT_OF_STOCK: set[str] = set(stock_df["product_id"]) if not stock_df.empty else set()



In [7]:
# Co-views: For every product, the top 20 products most viewed in the same sessions
coviews_sql = """
SELECT a.product_id AS p, b.product_id AS q, count() AS n
FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS a
INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS b
  ON a.session_id = b.session_id
WHERE a.product_id != b.product_id
GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p
"""
coviews_df = query(coviews_sql)
COVIEWS_DICT: dict[str, list[str]] = (
    coviews_df.groupby("p").q.apply(list).to_dict() if not coviews_df.empty else {}
)


In [8]:

# Active incoming recommendation requests queue
requests_sql = """
SELECT request_id, ts, user_id, context, recent_views
FROM shop.requests
ORDER BY ts DESC
LIMIT 100
"""
RECENT_REQUESTS: pd.DataFrame = query(requests_sql)



In [9]:

# Historical session logs (contains impressions, clicks, propensity scores)[cite: 1]
history_sessions_sql = """
SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
FROM shop.history_sessions
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_SESSIONS: pd.DataFrame = query(history_sessions_sql)

# Granular historical event logs (individual views and purchases for matrix factorization / item similarity)
history_events_sql = """
SELECT session_id, user_id, event_type, product_id, ts
FROM shop.history_events
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_EVENTS: pd.DataFrame = query(history_events_sql)


In [10]:
catalog = query("SELECT * FROM shop.catalog")
feedback_df = query(f"SELECT * FROM {TEAM}.feedback ORDER BY ts DESC LIMIT 1000")

In [11]:
# Summary of loaded variables
if __name__ == "__main__":
    print("\n--- Data Loaded Successfully ---")
    print(f"• BEST_SELLERS: {len(BEST_SELLERS)} products loaded")
    print(f"• OUT_OF_STOCK: {len(OUT_OF_STOCK)} items currently unavailable")
    print(f"• COVIEWS_DICT: {len(COVIEWS_DICT)} products have co-view maps")
    print(f"• RECENT_REQUESTS: {len(RECENT_REQUESTS)} live request rows fetched")
    print(f"• HISTORY_SESSIONS: {len(HISTORY_SESSIONS)} session rows fetched")
    print(f"• HISTORY_EVENTS: {len(HISTORY_EVENTS)} event rows fetched")


--- Data Loaded Successfully ---
• BEST_SELLERS: 4809 products loaded
• OUT_OF_STOCK: 36 items currently unavailable
• COVIEWS_DICT: 5000 products have co-view maps
• RECENT_REQUESTS: 100 live request rows fetched
• HISTORY_SESSIONS: 10000 session rows fetched
• HISTORY_EVENTS: 10000 event rows fetched


---

In [12]:
bestSellers = pd.DataFrame(BEST_SELLERS)
outOfStock = pd.DataFrame(OUT_OF_STOCK)
coviewsDict = COVIEWS_DICT
recentRequests = pd.DataFrame(RECENT_REQUESTS)
historySessions = pd.DataFrame(HISTORY_SESSIONS)
historyEvents = pd.DataFrame(HISTORY_EVENTS)
catalog = catalog
feedback = feedback_df


In [13]:
# ===== DATOS DE ENTRENAMIENTO (sin fuga del futuro) =====
# Las celdas de arriba (BEST_SELLERS, COVIEWS_DICT...) usan los 60 dias: sirven para SERVIR en vivo, pero para
# ENTRENAR habria fuga (incluirian la propia sesion y el futuro). Aqui se recalculan con datos ANTERIORES al corte.
import numpy as np

# CUT = el mismo corte que usa user_item.py para entrenar iALS: peticiones DESPUES, historial ANTES.
last_day = query("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())
print("CUT =", CUT)

BEST_TRAIN = query(f"""
SELECT product_id::String AS product_id, count() AS n FROM shop.history_events
WHERE event_type::String = 'purchase' AND ts::String < '{CUT}'
GROUP BY product_id ORDER BY n DESC""").product_id.tolist()

_cv = query(f"""
SELECT a.product_id AS p, b.product_id AS q, count() AS n
FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events
      WHERE event_type::String = 'view' AND ts::String < '{CUT}') AS a
INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events
      WHERE event_type::String = 'view' AND ts::String < '{CUT}') AS b ON a.session_id = b.session_id
WHERE a.product_id != b.product_id
GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
COVIEWS_TRAIN = _cv.groupby("p").q.apply(list).to_dict() if not _cv.empty else {}

# Peticiones posteriores al corte (1 de cada 4). Se quedan hasta N_POS con clic/compra y N_NEG sin clic,
# y 'sample_mult' dice cuantas peticiones representa cada una (asi las tasas siguen siendo las reales).
N_POS, N_NEG = 1500, 1500
pool = query(f"""
SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
FROM shop.history_sessions
WHERE ts::String >= '{CUT}' AND cityHash64(request_id::String) % 4 = 0""")
is_pos = pool.clicked.notna() | pool.purchased.notna()
pos, neg = pool[is_pos], pool[~is_pos]
pos_s = pos.sample(n=min(N_POS, len(pos)), random_state=42)
neg_s = neg.sample(n=min(N_NEG, len(neg)), random_state=42)
TRAIN_SESSIONS = pd.concat([
    pos_s.assign(sample_mult=len(pos) / max(len(pos_s), 1)),
    neg_s.assign(sample_mult=len(neg) / max(len(neg_s), 1)),
]).sort_values("ts").reset_index(drop=True)
print(f"{len(pool)} peticiones en el pool | muestra: {len(pos_s)} con clic + {len(neg_s)} sin clic")
print("tasa real de peticiones con clic:", round(len(pos) / len(pool), 4))


CUT = 2026-11-07
61751 peticiones en el pool | muestra: 1500 con clic + 1500 sin clic
tasa real de peticiones con clic: 0.0383


In [14]:
# ANTES: solo peticiones con clic -> tasa de clic falsa (10%). AHORA: muestra con positivos Y negativos.
sampled_interactions = TRAIN_SESSIONS
print(len(sampled_interactions), "peticiones |", int(sampled_interactions.clicked.notna().sum()), "con clic")


3000 peticiones | 1500 con clic


In [15]:
from featureConstructor import FeatureConstructor

In [16]:
import importlib
import featureConstructor
importlib.reload(featureConstructor)
from featureConstructor import FeatureConstructor

USE_IALS = False   # pasa a True cuando hayas reentrenado iALS con user_item.py corregido (y exista models/)

fc = FeatureConstructor(
    query_function=query,
    catalog=catalog,
    coview_dict=COVIEWS_TRAIN,      # solo datos anteriores al corte (sin fuga)
    best_sellers=BEST_TRAIN,
    out_of_stock=OUT_OF_STOCK,
    use_ials_features=USE_IALS,
)

# generate_xy descarga el historial de todos los usuarios de golpe (prefetch) y calcula las features.
X, y_click, y_buy, sample_weights = fc.generate_xy(
    interactions_df=sampled_interactions,
    history_sessions=TRAIN_SESSIONS,
)


Descargando historial (2746 usuarios):   0%|          | 0/10 [00:00<?, ?lote/s]

Generando features:   0%|          | 0/3000 [00:00<?, ?it/s]

In [17]:
output_df = X.assign(
    y_click=y_click,
    y_buy=y_buy,
    sample_weights=sample_weights,
    request_id=fc.last_meta.request_id.values,
    ts=fc.last_meta.ts.values,
    sample_mult=fc.last_meta.sample_mult.values,
)
output_df.to_csv("xy_dataset.csv", index=False)


In [18]:
m_ = fc.last_meta.sample_mult.values
print("Filas:", len(X), "| peticiones:", fc.last_meta.request_id.nunique())
print("Clicks en la muestra:", y_click.sum(), f"({y_click.mean():.1%} de las filas; esta tasa NO es la real, la muestra esta enriquecida)")
print("Buys en la muestra:  ", y_buy.sum())
print("Tasa de clic REAL por impresion (ponderada):", f"{np.average(y_click, weights=m_):.3%}")
print("Tasa de compra REAL por impresion (ponderada):", f"{np.average(y_buy, weights=m_):.3%}")


Filas: 30000 | peticiones: 3000
Clicks en la muestra: 1500 (5.0% de las filas; esta tasa NO es la real, la muestra esta enriquecida)
Buys en la muestra:   569
Tasa de clic REAL por impresion (ponderada): 0.383%
Tasa de compra REAL por impresion (ponderada): 0.145%


In [19]:
const = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
print("Columnas constantes (no aportan nada):", const)
print("Con muchos NaN:", X.isna().mean()[lambda s: s > 0.2].round(2).to_dict())


Columnas constantes (no aportan nada): ['is_new_user']
Con muchos NaN: {}


In [20]:
# ===== VALIDACION: corte temporal por peticion + ablacion (con / sin features de usuario) =====
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

meta = fc.last_meta.reset_index(drop=True)
reqs = meta.drop_duplicates("request_id").sort_values("ts")
cutoff = reqs.ts.iloc[int(len(reqs) * 0.7)]            # 70% antiguo -> entrena, 30% reciente -> valida
tr, va = (meta.ts < cutoff).values, (meta.ts >= cutoff).values

BASE_COLS = ["brand", "category", "subcategory", "device", "region", "page", "price_eur", "product_age_days",
             "description_len", "hour_of_day", "day_of_week", "is_weekend", "position", "session_length",
             "same_category_as_last_view", "same_brand_as_last_view", "price_diff_from_last_view", "is_new_user"]
BASE_COLS += [c for c in X.columns if c.startswith("coview_") or c in ("has_coview_signal", "time_since_last_trigger_sec")]
BASE_COLS = [c for c in BASE_COLS if c in X.columns]
ALL_COLS = list(X.columns)

def evaluate(cols, label):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
    m.fit(X.loc[tr, cols], y_click[tr], sample_weight=sample_weights[tr])
    p = m.predict_proba(X.loc[va, cols])[:, 1]
    auc = roc_auc_score(y_click[va], p, sample_weight=meta.sample_mult.values[va])
    Xv0 = X.loc[va, cols].copy()
    if "position" in cols:
        Xv0["position"] = 0                              # como al servir: sin efecto de posicion
    d = pd.DataFrame({"req": meta.request_id.values[va], "s": m.predict_proba(Xv0)[:, 1], "y": y_click[va]})
    d = d[d.groupby("req").y.transform("max") == 1]
    d["rk"] = d.groupby("req").s.rank(ascending=False, method="first")
    r = d[d.y == 1].rk
    print(f"{label:26s} AUC={auc:.4f} | top-3={(r <= 3).mean():.3f} (azar 0.300) | rango medio={r.mean():.2f} (azar 5.50) | {len(r)} peticiones")
    return m

m_base = evaluate(BASE_COLS, "producto/contexto/coview")
m_all = evaluate(ALL_COLS, "+ usuario" + (" + iALS" if USE_IALS else ""))
imp = pd.Series(m_all.feature_importances_, index=ALL_COLS).sort_values(ascending=False)
print("\nTop 15 features por importancia:\n", imp.head(15).to_string())


producto/contexto/coview   AUC=0.7511 | top-3=0.495 (azar 0.300) | rango medio=4.15 (azar 5.50) | 420 peticiones
+ usuario                  AUC=0.8397 | top-3=0.540 (azar 0.300) | rango medio=3.94 (azar 5.50) | 420 peticiones

Top 15 features por importancia:
 subcategory                    911
brand                          897
price_log_mean_last7d          205
position                       200
price_diff_from_last_view      194
region                         188
days_since_first_event         182
page_product_share             169
cat_share_Electronics          168
user_cat_share_of_candidate    160
cat_share_Fashion              157
buy_per_cart                   156
days_since_last_request        153
product_age_days               149
cart_per_view                  147


In [21]:
import pickle
final = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
final.fit(X, y_click, sample_weight=sample_weights)
#final.booster_.save_model("../model2.txt")   # includes the pandas category mapping

fc.set_schema(X)
pickle.dump({"feature_columns": fc.feature_columns,
             "column_dtypes": fc.column_dtypes,
             "category_dtypes": fc.category_dtypes},
            open("../schema.pkl", "wb"))

In [22]:
print("buys in sample:", y_buy.sum(), "| clicks:", y_click.sum())

def fit(y, mask=tr, cols=ALL_COLS):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
    m.fit(X.loc[mask, cols], y[mask], sample_weight=sample_weights[mask])
    return m

m_click, m_buy = fit(y_click), fit(y_buy)

Xv = X.loc[va, ALL_COLS].assign(position=0)
base = pd.DataFrame({"req": meta.request_id.values[va],
                     "pc": m_click.predict_proba(Xv)[:, 1], "pb": m_buy.predict_proba(Xv)[:, 1],
                     "yc": y_click[va], "yb": y_buy[va]})

def top3(score, ycol):
    d = base.assign(s=score)
    d = d[d.groupby("req")[ycol].transform("max") == 1]
    d["rk"] = d.groupby("req").s.rank(ascending=False, method="first")
    r = d[d[ycol] == 1].rk
    return (r <= 3).mean(), len(r)

for lam in [0, 1, 3, 10, 30]:       # p_buy is much smaller than p_click, so the weight needs to be large
    s = base.pc + lam * base.pb
    tc, nc = top3(s, "yc"); tb, nb = top3(s, "yb")
    print(f"lambda={lam:3}: clicks top-3={tc:.3f} ({nc} req) | buys top-3={tb:.3f} ({nb} req)")

buys in sample: 569 | clicks: 1500
lambda=  0: clicks top-3=0.540 (420 req) | buys top-3=0.605 (172 req)
lambda=  1: clicks top-3=0.545 (420 req) | buys top-3=0.605 (172 req)
lambda=  3: clicks top-3=0.550 (420 req) | buys top-3=0.622 (172 req)
lambda= 10: clicks top-3=0.583 (420 req) | buys top-3=0.640 (172 req)
lambda= 30: clicks top-3=0.562 (420 req) | buys top-3=0.616 (172 req)


In [ ]:
print(query(f"""
SELECT 
    toStartOfMinute(ts::DateTime64(3)) AS minute,
    served_by,
    model_version,
    count() AS n,
    min(latency_ms) AS min_ms,
    avg(latency_ms) AS avg_ms,
    max(latency_ms) AS max_ms
FROM {TEAM}.feedback
WHERE ts::DateTime64(3) > now() - INTERVAL 45 MINUTE
GROUP BY minute, served_by, model_version
ORDER BY minute DESC, served_by
"""))